In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Optional

import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
from src.gradcam import GradCAM
from src.activation import ActivationMapCollector
from src.model import FineTunedModel


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ----------------------------------------------------
# 1. ĐỊNH NGHĨA ĐƯỜNG DẪN VÀ BATCH SIZE
# ----------------------------------------------------
# Lấy đường dẫn từ ảnh của bạn
data_dir = 'data/imagenette' 
BATCH_SIZE = 1

# ----------------------------------------------------
# 2. ĐỊNH NGHĨA CÁC PHÉP BIẾN ĐỔI (TRANSFORMS)
# ----------------------------------------------------
# Đây là bước rất quan trọng.
# Model của bạn (giống AlexNet) cần ảnh đầu vào có kích thước
# cố định (ví dụ: 224x224) và đã được chuẩn hóa.

data_transform = transforms.Compose([
    # Resize ảnh về kích thước 224x224
    transforms.Resize((224, 224)),
    
    # (Tùy chọn) Thêm Augmentation để model học tốt hơn
    # transforms.RandomHorizontalFlip(), # Lật ảnh ngẫu nhiên
    
    # Chuyển ảnh (PIL Image) sang Tensor (PyTorch)
    # và scale giá trị pixel từ [0, 255] về [0.0, 1.0]
    transforms.ToTensor(),
    
    # Chuẩn hóa ảnh với Mean và Std của ImageNet
    # (Rất quan trọng nếu bạn dùng pre-trained weights)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ----------------------------------------------------
# 3. TẠO DATASET BẰNG IMAGEFOLDER
# ----------------------------------------------------
# Đây là "phép thuật" - nó tự động tìm các lớp
# từ tên thư mục con.
full_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=data_transform
)

print(f"Tìm thấy {len(full_dataset)} ảnh trong {len(full_dataset.classes)} lớp.")
print("Các lớp được tìm thấy:", full_dataset.classes)

# ----------------------------------------------------
# 4. TẠO DATALOADER
# ----------------------------------------------------
# DataLoader chịu trách nhiệm xáo trộn (shuffle),
# tạo batch, và tải dữ liệu song song.

data_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,

)

# ----------------------------------------------------
# 5. (TÙY CHỌN) KIỂM TRA
# ----------------------------------------------------
print("\nKiểm tra một batch từ DataLoader:")
try:
    # Lấy một batch đầu tiên
    images, labels = next(iter(data_loader))
    
    print(f"- Kích thước batch ảnh (Images shape): {images.shape}") 
    # Sẽ in ra: [64, 3, 224, 224] (Batch, Channels, Height, Width)
    
    print(f"- Kích thước batch nhãn (Labels shape): {labels.shape}") 
    # Sẽ in ra: [64]
    
except Exception as e:
    print(f"Lỗi khi tải dữ liệu: {e}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FineTunedModel(num_classes=10).to(device)
model.load_state_dict(torch.load('weights/finetune_weights.pth'))
target_layer = model.feature_extractor[5]


In [ ]:
collector = ActivationMapCollector(model, target_layer, device=device)
maps = collector.collect_maps(data_loader)


In [ ]:
len(maps)

In [ ]:
from src.hals_nnd import hals_nnd_correct
D, W, loss_history = hals_nnd_correct(
    maps, m=500, nsteps=20, batch_size=2000, device=device
)

In [ ]:
loss_history.keys()

In [ ]:
plt.plot(loss_history['total'])

In [ ]:
plt.plot(loss_history['reconstruction'])

In [ ]:
plt.plot(loss_history['regularization'])

In [ ]:
torch.save(D, 'weights/hals_nnd_D.pth')